# Triple-Frame Regions: CDS Context Classification

**Objective**: Load results from Notebook 1 and classify regions by their relationship to CDSs.

**IMPORTANT**: This notebook requires Notebook 1 to be run first!

**CDS Context Categories**:
1. **Includes CDS**: Triple-frame region contains the entire CDS
2. **Overlaps Upstream**: Region overlaps 5' end of CDS (upstream context)
3. **Overlaps Downstream**: Region overlaps 3' end of CDS (downstream context)
4. **CDS-Internal**: Region is entirely within the CDS
5. **No Overlap**: Region doesn't overlap any CDS
6. **No CDS Annotation**: Transcript has no annotated CDS

**Runtime**: ~1 minute (just classification, no re-analysis)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

print("Libraries loaded successfully")

## 1. Load Results from Notebook 1

In [ ]:
# Load the two analyses from Notebook 1
results_dir = Path('../results/triple_frame')

print("Loading results from Notebook 1...")

df_stop_free = pd.read_csv(results_dir / 'stop_free_triple_frame.tsv', sep='\t')
print(f"  Loaded {len(df_stop_free):,} stop-free regions")

df_atg_stop = pd.read_csv(results_dir / 'atg_stop_triple_frame.tsv', sep='\t')
print(f"  Loaded {len(df_atg_stop):,} ATG-to-STOP regions")

print("\nAll data loaded successfully!")

In [ ]:
def classify_cds_context(region_start, region_end, cds_start, cds_end):
    """
    Classify the relationship between a triple-frame region and the CDS
    using original category names, but with updated logic:
    
    - overlaps_upstream: region starts at CDS start or overlaps the 5' end
    - overlaps_downstream: region ends at CDS stop or overlaps 3' end
    - cds_internal: region entirely inside CDS
    - includes_cds: region spans entire CDS
    - no_overlap: region does not overlap CDS
    - no_cds_annotation: CDS coordinates missing
    """

    # No CDS annotation
    if pd.isna(cds_start) or pd.isna(cds_end):
        return 'no_cds_annotation'

    # No overlap
    if region_end <= cds_start or region_start >= cds_end:
        return 'no_overlap'

    # Entire CDS is covered
    if region_start <= cds_start and region_end >= cds_end:
        return 'includes_cds'

    # Starts exactly at CDS start → upstream overlap
    if region_start == cds_start:
        return 'overlaps_upstream'

    # Ends exactly at CDS stop → downstream overlap
    if region_end == cds_end:
        return 'overlaps_downstream'

    # Fully inside CDS
    if region_start > cds_start and region_end < cds_end:
        return 'cds_internal'

    # Partial overlaps that don't match exact boundaries
    # e.g., region_start < cds_start and region_end < cds_end → upstream overlap
    if region_start < cds_start and region_end < cds_end:
        return 'overlaps_upstream'

    # e.g., region_start > cds_start and region_end > cds_end → downstream overlap
    if region_start > cds_start and region_end > cds_end:
        return 'overlaps_downstream'

    # Fallback to include CDS if somehow spanning
    return 'includes_cds'


# Test the classification
test_cases = [
    (0, 500, 100, 400, 'includes_cds'),
    (200, 300, 100, 400, 'cds_internal'),
    (50, 200, 100, 400, 'overlaps_upstream'),
    (300, 500, 100, 400, 'overlaps_downstream'),
    (0, 50, 100, 400, 'no_overlap'),
    (0, 100, None, None, 'no_cds_annotation'),
]

print("Testing CDS context classification:")
for r_start, r_end, c_start, c_end, expected in test_cases:
    result = classify_cds_context(r_start, r_end, c_start, c_end)
    status = "✓" if result == expected else "✗"
    print(f"  {status} Region({r_start}-{r_end}) vs CDS({c_start}-{c_end}): {result}")

print("\nCDS context classification function ready!")

## 3. Apply CDS Context Classification to All Datasets

In [ ]:
print("\n" + "="*70)
print("APPLYING CDS CONTEXT CLASSIFICATION")
print("="*70)

# Apply classification to each dataset
for name, df in [('Stop-Free', df_stop_free), ('ATG-to-STOP', df_atg_stop)]:
    print(f"\n{name}:")
    df['cds_context'] = df.apply(
        lambda row: classify_cds_context(
            row['region_start'], row['region_end'],
            row['cds_start'], row['cds_end']
        ), axis=1
    )
    print(f"  ✓ Classified {len(df):,} regions")
    print(f"\n  Distribution:")
    print(df['cds_context'].value_counts().to_string().replace('\n', '\n  '))

print("\n" + "="*70)
print("Classification complete!")
print("="*70)

In [ ]:
print("\n" + "="*70)
print("SRD5A1 ANALYSIS WITH CDS CONTEXT")
print("="*70)

for analysis_name, df in [('Stop-Free', df_stop_free), ('ATG-to-STOP', df_atg_stop)]:
    print(f"\n{analysis_name}:")
    srd5a1 = df[df['gene_name'] == 'SRD5A1']
    
    if len(srd5a1) > 0:
        print(f"  ✅ Found {len(srd5a1)} regions")
        
        # Show CDS context breakdown
        print(f"\n  CDS context breakdown:")
        for context in srd5a1['cds_context'].unique():
            count = len(srd5a1[srd5a1['cds_context'] == context])
            print(f"    {context}: {count} regions")
        
        # Show best region
        best = srd5a1.iloc[0]
        percentile = 100 * (1 - best['rank'] / len(df))
        print(f"\n  Best region:")
        print(f"    Rank: #{best['rank']:,} / {len(df):,} ({percentile:.1f}th percentile)")
        print(f"    Length: {best['region_length_codons']} codons")
        print(f"    CDS context: {best['cds_context']}")
        if best['cds_overlap_codons'] > 0:
            print(f"    CDS overlap: {best['cds_overlap_codons']} codons")
    else:
        print("  ❌ Not found")

In [ ]:
# Save both analyses with CDS context
output_dir = Path('../results/triple_frame_cds_context')
output_dir.mkdir(parents=True, exist_ok=True)

df_stop_free.to_csv(output_dir / 'stop_free_with_cds_context.tsv', sep='\t', index=False)
print(f"\nSaved: {output_dir / 'stop_free_with_cds_context.tsv'}")

df_atg_stop.to_csv(output_dir / 'atg_stop_with_cds_context.tsv', sep='\t', index=False)
print(f"Saved: {output_dir / 'atg_stop_with_cds_context.tsv'}")

In [ ]:
# Create stacked bar chart showing CDS context distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
datasets = [
    (df_stop_free, 'Stop-Free', 'steelblue'),
    (df_atg_stop, 'ATG→STOP', 'forestgreen')
]
context_order = ['includes_cds', 'overlaps_upstream', 'overlaps_downstream', 'cds_internal', 'no_overlap', 'no_cds_annotation']
context_labels = ['Includes\nCDS', 'Overlaps\nUpstream', 'Overlaps\nDownstream', 'CDS\nInternal', 'No\nOverlap', 'No CDS\nAnnotation']
context_colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#95a5a6', '#34495e']

for idx, (df, title, base_color) in enumerate(datasets):
    ax = axes[idx]
    
    # Count by CDS context
    context_counts = df['cds_context'].value_counts()
    
    # Prepare data in correct order
    counts = [context_counts.get(ctx, 0) for ctx in context_order]
    percentages = [100 * c / len(df) for c in counts]
    
    # Create bar chart
    bars = ax.bar(range(len(context_order)), percentages, color=context_colors, edgecolor='black', alpha=0.8)
    
    ax.set_xlabel('CDS Context', fontsize=11)
    ax.set_ylabel('Percentage of Regions (%)', fontsize=11)
    ax.set_title(f'{title}\n(n={len(df):,})', fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(context_order)))
    ax.set_xticklabels(context_labels, rotation=0, ha='center', fontsize=8)
    ax.set_yscale('log')
    ax.set_ylim(0.1, max(percentages) * 2)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add percentage labels on bars
    for i, (bar, pct, count) in enumerate(zip(bars, percentages, counts)):
        if pct > 1:  # Only show label if >1%
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                   f'{pct:.1f}%\n(n={count:,})',
                   ha='center', va='bottom', fontsize=7, fontweight='bold')

plt.tight_layout()
fig_path = '../results/figures/cds_context_distribution.png'
Path(fig_path).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.savefig(fig_path.replace('.png', '.pdf'), bbox_inches='tight')
print(f"\nSaved figure: {fig_path}")
plt.show()